In [ ]:
import os, json, cv2
import numpy as np

import mediapipe as mp
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter

from sklearn.metrics import accuracy_score, classification_report

In [2]:
import pandas as pd

In [4]:
DATASET_PATH = r"D:\sign language\sign dataset"
CSV_PATH = os.path.join(DATASET_PATH, "dataset.csv")
OUTPUT_PATH = r"D:\sign language\output"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Training settings
MAX_FRAMES = 64
BATCH_SIZE = 8
EPOCHS = 10
LEARNING_RATE = 0.0001

# MediaPipe settings
NUM_HANDS = 2
HAND_LANDMARKS = 21
FACE_LANDMARKS = 468
POSE_LANDMARKS = 33

# Feature calculation
HAND_FEATURES = (NUM_HANDS * HAND_LANDMARKS * 3)  # 2 * 21 * 3 = 126
FACE_FEATURES = (FACE_LANDMARKS * 3)  # 468 * 3 = 1404
POSE_FEATURES = (POSE_LANDMARKS * 4)  # x, y, z, visibility -> 33 * 4 = 132
FEATURES = (HAND_FEATURES + FACE_FEATURES + POSE_FEATURES)

print("\nFeature configuration")
print("----------------------------")
print("Hand features :", HAND_FEATURES)
print("Face features :", FACE_FEATURES)
print("Pose features :", POSE_FEATURES)
print("Total features:", FEATURES)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", DEVICE)


Feature configuration
----------------------------
Hand features : 126
Face features : 1404
Pose features : 132
Total features: 1662

Device: cpu


##  Create Dataset CSV

In [ ]:
def create_csv():
    records = []

    # WORD dataset

    word_path = os.path.join(DATASET_PATH, "Word")

    if os.path.exists(word_path):
        for letter in os.listdir(word_path):
            letter_path = os.path.join(word_path, letter)

            for label in os.listdir(letter_path):
                label_path = os.path.join(letter_path, label)

                for file in os.listdir(label_path):
                    if file.lower().endswith((".jpg", ".jpeg", ".png")):
                        image_path = os.path.join(label_path, file)
                        records.append([file, "word", image_path , label])
                    

    # SENTENCE dataset
    sentence_path = os.path.join(DATASET_PATH, "sentence")

    if os.path.exists(sentence_path):
        for label in os.listdir(sentence_path):
            label_folder = os.path.join(sentence_path, label)

            for sample in os.listdir(label_folder):
                sample_folder = os.path.join(label_folder, sample)

                images = [
                    f for f in os.listdir(sample_folder)
                    if f.lower().endswith((".jpg", ".jpeg", ".png"))
                ]

                if images:
                    records.append([sample, "sentence", sample_folder, label])

    if not records:
        raise ValueError("No dataset found. Check DATASET_PATH.")

    df = pd.DataFrame(records, columns=["name", "type", "path", "label"])
    df.to_csv(CSV_PATH, index=False)

    print("\nCSV created:", CSV_PATH)
    print("Samples:", len(df))
    print("Classes:", df["label"].nunique())

    return df


df = create_csv()


CSV created:
D:\sign language\sign dataset\dataset.csv

Samples: 685
Classes: 101


##  Label Encoding

In [32]:
labels = sorted(df["label"].unique())
label_to_id = {label: i for i, label in enumerate(labels)}
id_to_label = {i: label for label, i in label_to_id.items()}
df["label_id"] = (df["label"].map(label_to_id))
NUM_CLASSES = len(labels)
print("\nNumber of classes:", NUM_CLASSES)

# Save label encoder
with open(os.path.join(OUTPUT_PATH, "labels.json"), "w", encoding="utf-8") as f:
    json.dump(label_to_id, f, indent=4)


Number of classes: 101




##  MediaPipe (Hand + Face + Pose)

In [33]:
mp_hands = mp.solutions.hands
mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose

# Hand model
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=NUM_HANDS, min_detection_confidence=0.5)

# Face model
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1,
                                  refine_landmarks=False, min_detection_confidence=0.5)

# Pose model
pose = mp_pose.Pose(static_image_mode=True, model_complexity=1,
                    enable_segmentation=False, min_detection_confidence=0.5)

##  Normalization 

In [34]:
def normalize_hand(hand):
    points = np.array([[p.x, p.y, p.z] for p in hand.landmark], dtype=np.float32)

    # Wrist becomes origin
    points = (points - points[0])

    # Scale normalization
    scale = np.max(np.linalg.norm(points, axis=1))
    if scale > 0:
        points = (points / scale)
    return points.flatten()


def normalize_face(face):
    points = np.array([[p.x, p.y, p.z] for p in face.landmark], dtype=np.float32)

    # Use nose region as approximate face center (landmark 1 = nose)
    center = points[1].copy()
    points = (points - center)

    # Scale normalization
    scale = np.max(np.linalg.norm(points, axis=1))
    if scale > 0:
        points = (points / scale)
    return points.flatten()


def normalize_pose(pose_landmarks):
    points = []
    for p in pose_landmarks.landmark:
        points.append([p.x, p.y, p.z, p.visibility])
    points = np.array(points, dtype=np.float32)

    # Use hips as body center (Left hip = 23, Right hip = 24)
    left_hip = points[23, :3]
    right_hip = points[24, :3]
    center = (left_hip + right_hip) / 2.0
    points[:, :3] -= center

    # Scale using body size
    scale = np.max(np.linalg.norm(points[:, :3], axis=1))
    if scale > 0:
        points[:, :3] /= scale
    return points.flatten()

In [35]:
def get_images(path, sample_type):
    # WORD
    if sample_type == "word":
        return [path]

    # SENTENCE
    images = []
    if os.path.isdir(path):
        for file in os.listdir(path):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                images.append(os.path.join(path, file))

    # SORT FRAMES
    images.sort()
    return images


def process_image(path):
    image = cv2.imread(path)

    # IMAGE ERROR
    if image is None:
        return np.zeros(FEATURES, dtype=np.float32)

    # BGR -> RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # MEDIAPIPE HAND / FACE / POSE
    hand_result = hands.process(image)
    face_result = face_mesh.process(image)
    pose_result = pose.process(image)

    # CREATE FEATURE ARRAYS
    hand_features = np.zeros(HAND_FEATURES, dtype=np.float32)
    face_features = np.zeros(FACE_FEATURES, dtype=np.float32)
    pose_features = np.zeros(POSE_FEATURES, dtype=np.float32)

    # HAND FEATURES
    if hand_result.multi_hand_landmarks:
        left = None
        right = None
        for i, hand in enumerate(hand_result.multi_hand_landmarks):
            data = normalize_hand(hand)

            # Identify left/right hand
            side = (hand_result.multi_handedness[i].classification[0].label)
            if side == "Left":
                left = data
            else:
                right = data

        # LEFT HAND
        if left is not None:
            hand_features[:63] = left

        # RIGHT HAND
        if right is not None:
            hand_features[63:126] = right

    # FACE FEATURES
    if face_result.multi_face_landmarks:
        face = (face_result.multi_face_landmarks[0])
        face_features = normalize_face(face)

    # POSE FEATURES
    if pose_result.pose_landmarks:
        pose_features = normalize_pose(pose_result.pose_landmarks)

    # COMBINE ALL FEATURES
    features = np.concatenate([hand_features, face_features, pose_features])
    return features.astype(np.float32)


def extract_sample(path, sample_type):
    images = get_images(path, sample_type)
    if not images:
        return None

    # LIMIT TO 64 FRAMES
    if len(images) > MAX_FRAMES:
        indexes = np.linspace(0, len(images) - 1, MAX_FRAMES).astype(int)
        images = [images[i] for i in indexes]

    # EXTRACT FEATURES
    features = []
    for image in images:
        features.append(process_image(image))
    features = np.array(features, dtype=np.float32)

    # PADDING
    if len(features) < MAX_FRAMES:
        padding = np.zeros((MAX_FRAMES - len(features), FEATURES), dtype=np.float32)
        features = np.vstack([features, padding])

    return features[:MAX_FRAMES]

## 11. Create X and y

In [36]:
X = []
y = []

print("\nExtracting Hands + Face + Pose features...")

for index, row in df.iterrows():
    features = extract_sample(row["path"], row["type"])
    if features is not None:
        X.append(features)
        y.append(row["label_id"])

    # Progress
    if (index + 1) % 10 == 0:
        print(f"Processed {index + 1}/{len(df)}")

# Convert to numpy
X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)

print("\nFeature extraction complete.")
print("X shape:", X.shape)
print("Y shape:", y.shape)
print("\nExpected format:")
print("(samples, 64, 1662)")

# Save features
np.save(os.path.join(OUTPUT_PATH, "X.npy"), X)
np.save(os.path.join(OUTPUT_PATH, "y.npy"), y)


Extracting Hands + Face + Pose features...
Processed 10/685
Processed 20/685
Processed 30/685
Processed 40/685
Processed 50/685
Processed 60/685
Processed 70/685
Processed 80/685
Processed 90/685
Processed 100/685
Processed 110/685
Processed 120/685
Processed 130/685
Processed 140/685
Processed 150/685
Processed 160/685
Processed 170/685
Processed 180/685
Processed 190/685
Processed 200/685
Processed 210/685
Processed 220/685
Processed 230/685
Processed 240/685
Processed 250/685
Processed 260/685
Processed 270/685
Processed 280/685
Processed 290/685
Processed 300/685
Processed 310/685
Processed 320/685
Processed 330/685
Processed 340/685
Processed 350/685
Processed 360/685
Processed 370/685
Processed 380/685
Processed 390/685
Processed 400/685
Processed 410/685
Processed 420/685
Processed 430/685
Processed 440/685
Processed 450/685
Processed 460/685
Processed 470/685
Processed 480/685
Processed 490/685
Processed 500/685
Processed 510/685
Processed 520/685
Processed 530/685
Processed 5

In [ ]:


# Count samples in each class
class_counts = Counter(y)

print("Total samples:", len(y))
print("Total classes:", len(class_counts))

print("\nClasses with fewer than 2 samples:")
for label, count in sorted(class_counts.items()):
    if count < 2:
        print(f"Class {label}: {count} sample(s)")

Total samples: 685
Total classes: 101

Classes with fewer than 2 samples:
Class 1: 1 sample(s)
Class 51: 1 sample(s)


In [49]:
X = np.asarray(X)
y = np.asarray(y)

print(len(X) ,len(y)) 


685 685


In [ ]:


# Check dataset sizes
assert len(X) == len(y), "X and y have different sample counts!"

# 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

# 15% validation, 15% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

print("Total:", len(X))
print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Total: 685
Train: 479
Validation: 103
Test: 103


In [55]:
'''
# 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42,
    stratify=y if len(np.unique(y)) > 1 else None
)

#  divide temporary data into validation and test

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42,
    stratify=y_temp if len(np.unique(y_temp)) > 1 else None
)
'''

'\n# 70% training, 30% temporary\nX_train, X_temp, y_train, y_temp = train_test_split(\n    X, y, test_size=0.30, random_state=42,\n    stratify=y if len(np.unique(y)) > 1 else None\n)\n\n#  divide temporary data into validation and test\n\nX_val, X_test, y_val, y_test = train_test_split(\n    X_temp, y_temp, test_size=0.50, random_state=42,\n    stratify=y_temp if len(np.unique(y_temp)) > 1 else None\n)\n'

In [56]:


class SignDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]



train_dataset = SignDataset(X_train, y_train)
val_dataset   = SignDataset(X_val, y_val)
test_dataset  = SignDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

print("\nDataset split:")
print(f"Training:   {len(X_train) / len(X) * 100:.2f}%")
print(f"Validation: {len(X_val) / len(X) * 100:.2f}%")
print(f"Testing:    {len(X_test) / len(X) * 100:.2f}%")

Training samples: 479
Validation samples: 103
Testing samples: 103

Dataset split:
Training:   69.93%
Validation: 15.04%
Testing:    15.04%


## Transformer Model

In [57]:
class SignTransformer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # INPUT PROJECTION: 1662 -> 256
        self.input_layer = nn.Linear(FEATURES, 256)

        # POSITIONAL ENCODING
        self.positional_embedding = nn.Parameter(torch.zeros(1, MAX_FRAMES, 256))

        # TRANSFORMER ENCODER
        encoder = (nn.TransformerEncoderLayer(d_model=256, nhead=8, dim_feedforward=512,
                                              dropout=0.1, batch_first=True, norm_first=True))
        self.transformer = (nn.TransformerEncoder(encoder, num_layers=4))

        # NORMALIZATION
        self.norm = nn.LayerNorm(256)

        # CLASSIFIER
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes))

    def forward(self, x):
        # Feature projection
        x = self.input_layer(x)

        # Add temporal positional information
        x = (x + self.positional_embedding[:, :x.size(1), :])

        # Transformer
        x = self.transformer(x)

        # Global temporal pooling
        x = x.mean(dim=1)

        # Normalization
        x = self.norm(x)

        # Classification
        return self.classifier(x)


model = SignTransformer(NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

print("\nModel created.")
print("Input features:", FEATURES)
print("Classes:", NUM_CLASSES)


Model created.
Input features: 1662
Classes: 101


C:\Users\sunee\AppData\Local\Temp\ipykernel_20344\1426559057.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = (nn.TransformerEncoder(encoder, num_layers=4))


##  Training

In [58]:
best_accuracy = 0.0

print("\nTraining started...\n")

for epoch in range(EPOCHS):

    # TRAIN
    model.train()
    train_loss = 0.0
    train_correct = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += (loss.item())
        train_correct += (output.argmax(1) == y_batch).sum().item()

    train_accuracy = (train_correct / len(y_train))

    # VALIDATION
    model.eval()
    val_correct = 0
    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            output = model(X_batch)
            loss = criterion(output, y_batch)
            val_loss += (loss.item())
            val_correct += (output.argmax(1) == y_batch).sum().item()

    val_accuracy = (val_correct / len(y_val))

    # PRINT RESULTS
    print(f"Epoch {epoch + 1}/{EPOCHS} | "
          f"Train Loss: {train_loss / len(train_loader):.4f} | "
          f"Train Acc: {train_accuracy * 100:.2f}% | "
          f"Val Loss: {val_loss / len(val_loader):.4f} | "
          f"Val Acc: {val_accuracy * 100:.2f}%")

    # SAVE BEST MODEL
    if val_accuracy > best_accuracy:
        best_accuracy = (val_accuracy)
        torch.save({
            "model": model.state_dict(),
            "classes": label_to_id,
            "num_classes": NUM_CLASSES,
            "features": FEATURES,
            "max_frames": MAX_FRAMES,
            "hand_features": HAND_FEATURES,
            "face_features": FACE_FEATURES,
            "pose_features": POSE_FEATURES
        }, os.path.join(OUTPUT_PATH, "sign_transformer_best.pth"))
        print("  Best model saved!")


Training started...

Epoch 1/10 | Train Loss: 4.6634 | Train Acc: 1.04% | Val Loss: 4.6378 | Val Acc: 0.97%
  Best model saved!
Epoch 2/10 | Train Loss: 4.6260 | Train Acc: 1.04% | Val Loss: 4.6274 | Val Acc: 0.97%
Epoch 3/10 | Train Loss: 4.6278 | Train Acc: 0.42% | Val Loss: 4.6226 | Val Acc: 0.97%
Epoch 4/10 | Train Loss: 4.6187 | Train Acc: 1.25% | Val Loss: 4.6292 | Val Acc: 0.97%
Epoch 5/10 | Train Loss: 4.6136 | Train Acc: 1.67% | Val Loss: 4.6410 | Val Acc: 0.97%
Epoch 6/10 | Train Loss: 4.6009 | Train Acc: 1.25% | Val Loss: 4.6471 | Val Acc: 0.97%
Epoch 7/10 | Train Loss: 4.5961 | Train Acc: 1.88% | Val Loss: 4.6411 | Val Acc: 0.97%
Epoch 8/10 | Train Loss: 4.5794 | Train Acc: 1.25% | Val Loss: 4.6485 | Val Acc: 0.00%
Epoch 9/10 | Train Loss: 4.5753 | Train Acc: 1.46% | Val Loss: 4.6479 | Val Acc: 0.97%
Epoch 10/10 | Train Loss: 4.5621 | Train Acc: 1.46% | Val Loss: 4.6485 | Val Acc: 0.97%


## Save

In [ ]:
torch.save({
    "model": model.state_dict(),
    "classes": label_to_id,
    "num_classes": NUM_CLASSES,
    "features": FEATURES,
    "max_frames": MAX_FRAMES,
    "hand_features": HAND_FEATURES,
    "face_features": FACE_FEATURES,
    "pose_features": POSE_FEATURES
}, os.path.join(OUTPUT_PATH, "sign_transformer_final.pth"))

print("\nFinal model saved.")


Final model saved.


## Test 

In [60]:
len(X_test)

103

In [ ]:
model.eval()

sample = torch.tensor(X_test[1], dtype=torch.float32).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    output = model(sample)
    prediction = (output.argmax(1).item())

print("\nTest prediction:")
print("Predicted:", id_to_label[prediction])
print("Actual:", id_to_label[y_test[0]])
print("\nBest validation accuracy:", f"{best_accuracy * 100:.2f}%")
print("\nTraining complete!")

AttributeError: 'str' object has no attribute 'eval'